# Pilot 4 on the pod

One cell. Three rising beeps = done, two low beeps = stopped (scroll up). The recordings stay on the network volume, so later rounds do not need to record again.


In [ ]:
# ===== Pilot 4: record Qwen3-14B's insides and run the exploration (about 75 minutes) =====
# Pod: a 48 GB GPU (L40S or RTX A6000), Secure Cloud, with the network volume attached (200 GB, at /workspace).
# Results are written to files; the end of the log shows only progress, not findings.
# At the three rising beeps: download /workspace/pilot4_results.tar.gz and send it to Claude.
import os, subprocess, json, numpy as np
from IPython.display import Audio, display

REPO = "/workspace/llm-value-trade-offs"
P4 = f"{REPO}/pilot4-value-code"
os.environ["HF_HOME"] = "/workspace/hf"          # model cache on the network volume

def beep(ok=True):
    """Three rising beeps = done. Two low beeps = something failed."""
    sr = 22050
    t = np.linspace(0, 0.22, int(sr * 0.22), False)
    gap = np.zeros(int(sr * 0.08))
    freqs = [523, 659, 784] if ok else [220, 165]
    display(Audio(np.concatenate([np.concatenate([0.5 * np.sin(2 * np.pi * f * t), gap]) for f in freqs]),
                  rate=sr, autoplay=True))

def run(cmd, log_file=None):
    print(f"\n>>> {cmd}", flush=True)
    if log_file:
        cmd = f"set -o pipefail; ({cmd}) 2>&1 | tee {log_file}"
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"Failed: {cmd}")

def run_all(steps):
    try:
        for s in steps:
            run(*s) if isinstance(s, tuple) else run(s)
    except Exception as e:
        print(f"\n*** STOPPED: {e}")
        beep(False)
        return False
    beep(True)
    return True

run_all([
    f"cd /workspace && (test -d {REPO} || git clone https://github.com/JasperBlank/llm-value-trade-offs.git)",
    f"cd {REPO} && git fetch -q origin main && git reset -q --hard origin/main && git log -1 --format='Running commit %h: %s'",
    f"cd {P4} && pip install -q -r requirements.txt",
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader && df -h /workspace | tail -1",
    # A two-minute test of the whole pipeline on a small model, before the big run.
    f"cd {P4} && python record.py --quick && python record.py --quick --random && python probes.py --quick "
    f"&& python probes.py --quick --random && python neurons.py --quick && python compare.py --quick && python steer.py --quick",
    # The real run.
    (f"cd {P4} && python record.py", "/workspace/log_record.txt"),
    (f"cd {P4} && python record.py --random", "/workspace/log_record_random.txt"),
    (f"cd {P4} && python probes.py > /workspace/log_probes.txt 2>&1 && echo probes done", None),
    (f"cd {P4} && python probes.py --random > /workspace/log_probes_random.txt 2>&1 && echo control probes done", None),
    (f"cd {P4} && python neurons.py > /workspace/log_neurons.txt 2>&1 && python neurons.py --random >> /workspace/log_neurons.txt 2>&1 && echo neurons done", None),
    (f"cd {P4} && python compare.py > /workspace/log_compare.txt 2>&1 && echo comparison done", None),
    (f"cd {P4} && python steer.py > /workspace/log_steer.txt 2>&1 && echo steering done", None),
    f"cd /workspace && tar czf pilot4_results.tar.gz -C {P4}/results real real_random -C /workspace log_record.txt log_record_random.txt "
    f"log_probes.txt log_probes_random.txt log_neurons.txt log_compare.txt log_steer.txt && ls -la /workspace/pilot4_results.tar.gz",
])
